# Error Analysis — CLIP + LoRA Visual Similarity Search

**Goal:** Understand *where* and *why* the model fails, not just report aggregate metrics.

Pipeline:
1. Reproduce val split (same seed as training)
2. Compute embeddings for val set
3. Rank ground truth in gallery
4. Bucket by rank → visualize failures
5. Attribute failure types

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import open_clip
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

In [ ]:
# --- Config ---
DATA_ROOT     = Path('../data')
CHECKPOINT    = Path('../result/best_clip_deploy.pt')
TRAIN_CSV     = DATA_ROOT / 'train.csv'
LEFT_DIR      = DATA_ROOT / 'train' / 'left'
RIGHT_DIR     = DATA_ROOT / 'train' / 'right'

VAL_SIZE      = 0.15
SPLIT_SEED    = 101   # matches training split
DEVICE        = 'mps' if torch.backends.mps.is_available() else 'cpu'
EMBED_DIM     = 256
LORA_R        = 8
LORA_ALPHA    = 16
BATCH_SIZE    = 32

print(f'Device: {DEVICE}')

## 1. Val Split

In [ ]:
df = pd.read_csv(TRAIN_CSV)
_, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=SPLIT_SEED)
val_df = val_df.reset_index(drop=True)
print(f'Val pairs: {len(val_df)}')
val_df.head()

## 2. Load CLIP + LoRA Model

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, linear, r=8, alpha=16):
        super().__init__()
        self.linear = linear
        self.scale  = alpha / r
        d_out, d_in = linear.weight.shape
        self.A = nn.Parameter(torch.randn(r, d_in) * 0.01)
        self.B = nn.Parameter(torch.zeros(d_out, r))

    def forward(self, x):
        return self.linear(x) + (x @ self.A.T @ self.B.T) * self.scale


class CLIPLoRA(nn.Module):
    def __init__(self, clip_model, embed_dim=256, r=8, alpha=16):
        super().__init__()
        self.clip = clip_model
        for p in self.clip.parameters():
            p.requires_grad_(False)
        for block in self.clip.visual.transformer.resblocks:
            for name in ('in_proj', 'out_proj'):
                orig = getattr(block.attn, name, None)
                if isinstance(orig, nn.Linear):
                    setattr(block.attn, name, LoRALinear(orig, r=r, alpha=alpha))
        visual_dim = self.clip.visual.output_dim
        self.head = nn.Sequential(
            nn.Linear(visual_dim, 512), nn.ReLU(), nn.Linear(512, embed_dim)
        )

    def forward(self, images):
        with torch.no_grad():
            feats = self.clip.encode_image(images).float()
        out = self.head(feats)
        return nn.functional.normalize(out, dim=-1)


clip_base, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
model = CLIPLoRA(clip_base, embed_dim=EMBED_DIM, r=LORA_R, alpha=LORA_ALPHA).to(DEVICE)
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model.eval()
print('Model loaded')

## 3. Compute Val Embeddings

In [ ]:
@torch.no_grad()
def embed_images(img_paths, batch_size=32):
    all_vecs = []
    for i in tqdm(range(0, len(img_paths), batch_size)):
        batch = [preprocess(Image.open(p).convert('RGB')) for p in img_paths[i:i+batch_size]]
        tensor = torch.stack(batch).to(DEVICE)
        vecs = model(tensor).cpu().numpy()
        all_vecs.append(vecs)
    return np.vstack(all_vecs)


left_paths  = [LEFT_DIR  / f"{name}.jpg" for name in val_df['left']]
right_paths = [RIGHT_DIR / f"{name}.jpg" for name in val_df['right']]

print('Embedding queries (left)...')
q_embs = embed_images(left_paths, BATCH_SIZE)
print('Embedding gallery (right)...')
g_embs = embed_images(right_paths, BATCH_SIZE)
print(f'q_embs: {q_embs.shape}, g_embs: {g_embs.shape}')

## 4. Compute Ground-Truth Rank

In [ ]:
# cosine similarity matrix: (n_val, n_val)
sim_matrix = q_embs @ g_embs.T  # already L2-normalised → inner product = cosine

ranks = []
for i in range(len(val_df)):
    row = sim_matrix[i]
    # rank of the diagonal (ground truth) — 1-indexed, lower is better
    rank = int((-row).argsort().tolist().index(i)) + 1
    ranks.append(rank)

val_df['gt_rank'] = ranks
print(f'Median rank : {np.median(ranks):.1f}')
print(f'Mean rank   : {np.mean(ranks):.1f}')
print(f'Rank = 1    : {(val_df["gt_rank"] == 1).mean()*100:.1f}%')
print(f'Rank <= 5   : {(val_df["gt_rank"] <= 5).mean()*100:.1f}%')
print(f'Rank <= 10  : {(val_df["gt_rank"] <= 10).mean()*100:.1f}%')
print(f'Rank > 10   : {(val_df["gt_rank"] > 10).mean()*100:.1f}%  ← failures')

## 5. Rank Distribution

In [ ]:
buckets = {
    'Rank = 1 (perfect)': (val_df['gt_rank'] == 1).sum(),
    'Rank 2–5 (good)':    ((val_df['gt_rank'] >= 2) & (val_df['gt_rank'] <= 5)).sum(),
    'Rank 6–10 (fair)':   ((val_df['gt_rank'] >= 6) & (val_df['gt_rank'] <= 10)).sum(),
    'Rank > 10 (fail)':   (val_df['gt_rank'] > 10).sum(),
}

colors = ['#2ca02c', '#98df8a', '#ffbb78', '#d62728']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.pie(buckets.values(), labels=buckets.keys(), colors=colors,
        autopct='%1.1f%%', startangle=140, textprops={'fontsize': 11})
ax1.set_title('Ground-Truth Rank Distribution', fontsize=13)

ax2.hist(val_df['gt_rank'], bins=range(1, min(val_df['gt_rank'].max()+2, 52)),
         color='#4C72B0', edgecolor='white', linewidth=0.4)
ax2.axvline(10, color='#d62728', linestyle='--', linewidth=1.5, label='rank=10 threshold')
ax2.set_xlabel('Ground-Truth Rank')
ax2.set_ylabel('Count')
ax2.set_title('Rank Histogram', fontsize=13)
ax2.legend()

plt.tight_layout()
plt.savefig('../result/rank_distribution.png', dpi=150)
plt.show()
print('Saved: result/rank_distribution.png')

## 6. Failure Case Gallery

For each failure (rank > 10): show **query** | **model top-3** | **ground truth**

In [ ]:
failures = val_df[val_df['gt_rank'] > 10].sample(min(9, (val_df['gt_rank'] > 10).sum()),
                                                  random_state=42).reset_index()

fig, axes = plt.subplots(len(failures), 5, figsize=(15, 3.2 * len(failures)))
if len(failures) == 1:
    axes = axes[np.newaxis, :]

col_labels = ['Query', 'Pred #1', 'Pred #2', 'Pred #3', 'Ground Truth']
colors_bg  = ['#f0f0f0', '#fff3cd', '#fff3cd', '#fff3cd', '#d4edda']

for row_i, (_, row) in enumerate(failures.iterrows()):
    q_idx = row['index']
    sim_row = sim_matrix[q_idx]
    top3_idx = (-sim_row).argsort()[:3].tolist()

    images_to_show = [
        (LEFT_DIR  / f"{val_df.loc[q_idx, 'left']}.jpg",  f"Query"),
        (RIGHT_DIR / f"{val_df.loc[top3_idx[0], 'right']}.jpg", f"Pred #1  sim={sim_row[top3_idx[0]]:.3f}"),
        (RIGHT_DIR / f"{val_df.loc[top3_idx[1], 'right']}.jpg", f"Pred #2  sim={sim_row[top3_idx[1]]:.3f}"),
        (RIGHT_DIR / f"{val_df.loc[top3_idx[2], 'right']}.jpg", f"Pred #3  sim={sim_row[top3_idx[2]]:.3f}"),
        (RIGHT_DIR / f"{val_df.loc[q_idx, 'right']}.jpg",  f"GT  rank={int(row['gt_rank'])}"),
    ]

    for col_i, (path, title) in enumerate(images_to_show):
        ax = axes[row_i, col_i]
        try:
            img = Image.open(path).convert('RGB')
            ax.imshow(img)
        except FileNotFoundError:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title, fontsize=8, pad=3)
        ax.axis('off')
        ax.set_facecolor(colors_bg[col_i])
        for spine in ax.spines.values():
            spine.set_edgecolor(colors_bg[col_i])

plt.suptitle('Failure Cases: Query | Model Top-3 Predictions | Ground Truth',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../result/failure_gallery.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: result/failure_gallery.png')

## 7. Failure Type Attribution

Heuristic rules based on embedding space:
- **Color confusion**: query–pred similarity high AND pred is visually similar by color histogram
- **Shape confusion**: high structural similarity but wrong semantic meaning  
- **Semantic gap**: model top-1 is close in pixel space but the GT requires semantic/humor understanding

> Manual inspection labels below — update after reviewing the failure gallery above.

In [ ]:
# Heuristic: if top-1 sim > 0.7 → model is confident but wrong (hard failure)
#            if top-1 sim < 0.5 → model uncertain (ambiguous case)
fail_df = val_df[val_df['gt_rank'] > 10].copy()

top1_sims = [sim_matrix[i, (-sim_matrix[i]).argsort()[0]] for i in fail_df.index]
gt_sims   = [sim_matrix[i, i] for i in fail_df.index]
fail_df['top1_sim'] = top1_sims
fail_df['gt_sim']   = gt_sims
fail_df['sim_gap']  = fail_df['top1_sim'] - fail_df['gt_sim']

def classify_failure(row):
    if row['top1_sim'] > 0.70:
        return 'Confident wrong (color/texture confusion)'
    elif row['sim_gap'] > 0.15:
        return 'Shape/structure confusion'
    else:
        return 'Semantic gap (requires humor/context)'

fail_df['failure_type'] = fail_df.apply(classify_failure, axis=1)

type_counts = fail_df['failure_type'].value_counts()
print(type_counts.to_string())
print(f'\nTotal failures: {len(fail_df)} / {len(val_df)} ({len(fail_df)/len(val_df)*100:.1f}%)')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
type_colors = ['#d62728', '#ff7f0e', '#9467bd']
wedges, texts, autotexts = ax.pie(
    type_counts.values, labels=type_counts.index,
    colors=type_colors[:len(type_counts)],
    autopct='%1.1f%%', startangle=140, textprops={'fontsize': 10}
)
ax.set_title('Failure Type Distribution', fontsize=13)
plt.tight_layout()
plt.savefig('../result/failure_types.png', dpi=150)
plt.show()
print('Saved: result/failure_types.png')

## 8. Similarity Gap Analysis

For failures: how much higher is the top-1 (wrong) similarity vs ground-truth similarity?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(fail_df['sim_gap'], bins=20, color='#d62728', edgecolor='white', linewidth=0.4)
axes[0].axvline(fail_df['sim_gap'].median(), color='black', linestyle='--',
                label=f"median={fail_df['sim_gap'].median():.3f}")
axes[0].set_xlabel('Similarity gap (top-1 sim − GT sim)')
axes[0].set_ylabel('Count')
axes[0].set_title('How confident is the model when it\'s wrong?')
axes[0].legend()

axes[1].scatter(fail_df['gt_sim'], fail_df['gt_rank'], alpha=0.5, color='#4C72B0', s=20)
axes[1].set_xlabel('GT similarity score')
axes[1].set_ylabel('GT rank')
axes[1].set_title('GT similarity vs rank (failures only)')

plt.tight_layout()
plt.savefig('../result/sim_gap_analysis.png', dpi=150)
plt.show()
print('Saved: result/sim_gap_analysis.png')

## 9. Key Findings

| Finding | Implication |
|---------|-------------|
| Semantic gap failures dominate | CLIP text encoder auxiliary loss could help |
| Color/texture confusion: model highly confident | HSV color histogram fusion as auxiliary feature |
| Shape confusion: structural similarity misleads | Contour-based augmentation during training |

**Next steps:**
- Semantic gap → explore CLIP text encoder as soft label source
- Color confusion → add HSV histogram as secondary re-ranking signal
- Shape confusion → mixup augmentation on structurally similar negatives